# AdventureWorks Star Schema Load

Reads the raw headerless TSVs (`.csv` files) from the rawdata volume, applies the column names from the `CREATE TABLE` statements in `instawdb.sql`, and saves them as fact/dim tables in `adventure_works.silver`.

Notes:
* Raw files are **tab-separated** and have **no header row** (so `header=False`).
* Unity Catalog stores table names in lowercase, so `dim_PurchaseOrderDetail` is saved as `dim_purchaseorderdetail`.

In [0]:
# Configuration — raw files are tab-separated (.csv extension) with no header row
source_volume = "/Volumes/adventure_works/default/rawdata"
target_schema = "adventure_works.silver"

# CSV file -> (target table, column names in CREATE TABLE order from instawdb.sql)
table_configs = {
    "SalesOrderDetail.csv": (
        "fact_sales",
        ["SalesOrderID", "SalesOrderDetailID", "CarrierTrackingNumber", "OrderQty",
         "ProductID", "SpecialOfferID", "UnitPrice", "UnitPriceDiscount", "LineTotal",
         "rowguid", "ModifiedDate"],
    ),
    "Product.csv": (
        "dim_product",
        ["ProductID", "Name", "ProductNumber", "MakeFlag", "FinishedGoodsFlag", "Color",
         "SafetyStockLevel", "ReorderPoint", "StandardCost", "ListPrice", "Size",
         "SizeUnitMeasureCode", "WeightUnitMeasureCode", "Weight", "DaysToManufacture",
         "ProductLine", "Class", "Style", "ProductSubcategoryID", "ProductModelID",
         "SellStartDate", "SellEndDate", "DiscontinuedDate", "rowguid", "ModifiedDate"],
    ),
    "SalesTerritory.csv": (
        "dim_sales_territory",
        ["TerritoryID", "Name", "CountryRegionCode", "Group", "SalesYTD", "SalesLastYear",
         "CostYTD", "CostLastYear", "rowguid", "ModifiedDate"],
    ),
    "PurchaseOrderDetail.csv": (
        "dim_PurchaseOrderDetail",
        ["PurchaseOrderID", "PurchaseOrderDetailID", "DueDate", "OrderQty", "ProductID",
         "UnitPrice", "LineTotal", "ReceivedQty", "RejectedQty", "StockedQty", "ModifiedDate"],
    ),
    "PurchaseOrderHeader.csv": (
        "dim_PurchaseOrderHeader",
        ["PurchaseOrderID", "RevisionNumber", "Status", "EmployeeID", "VendorID",
         "ShipMethodID", "OrderDate", "ShipDate", "SubTotal", "TaxAmt", "Freight",
         "TotalDue", "ModifiedDate"],
    ),
    "Vendor.csv": (
        "dim_Vendor",
        ["BusinessEntityID", "AccountNumber", "Name", "CreditRating", "PreferredVendorStatus",
         "ActiveFlag", "PurchasingWebServiceURL", "ModifiedDate"],
    ),
}

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_schema}")

for csv_name, (target_table, columns) in table_configs.items():
    source_path = f"{source_volume}/{csv_name}"

    # Raw files are headerless TSVs: read without a header, then apply the
    # column names taken from the CREATE TABLE statements in instawdb.sql.
    df = (
        spark.read
        .option("header", False)
        .option("sep", "\t")
        .option("inferSchema", True)
        .csv(source_path)
        .toDF(*columns)
    )

    rows_written = df.count()

    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{target_schema}.{target_table}"))

    print(f"{target_table}: {rows_written:,} rows written to {target_schema}.{target_table}")

fact_sales: 121,317 rows written to adventure_works.silver.fact_sales
dim_product: 504 rows written to adventure_works.silver.dim_product
dim_sales_territory: 10 rows written to adventure_works.silver.dim_sales_territory
dim_PurchaseOrderDetail: 8,845 rows written to adventure_works.silver.dim_PurchaseOrderDetail
dim_PurchaseOrderHeader: 4,012 rows written to adventure_works.silver.dim_PurchaseOrderHeader
dim_Vendor: 104 rows written to adventure_works.silver.dim_Vendor


In [0]:
# Row counts from the bronze layer, to confirm nothing was lost or duplicated
expected_rows = {
    "fact_sales": 121317,
    "dim_product": 504,
    "dim_sales_territory": 10,
    "dim_PurchaseOrderDetail": 8845,
    "dim_PurchaseOrderHeader": 4012,
    "dim_Vendor": 104,
}

verification_rows = []
for csv_name, (target_table, columns) in table_configs.items():
    loaded = spark.table(f"{target_schema}.{target_table}")
    verification_rows.append((
        target_table,
        expected_rows[target_table],
        loaded.count(),
        loaded.columns == columns,
    ))

verification_df = spark.createDataFrame(
    verification_rows,
    schema="table string, expected_rows long, loaded_rows long, columns_match boolean",
)
display(verification_df)

bad = verification_df.filter("loaded_rows != expected_rows OR NOT columns_match")
assert bad.count() == 0, f"Mismatches found: {bad.collect()}"
print("Success: all 6 fact/dim tables loaded with the correct column names.")

table,expected_rows,loaded_rows,columns_match
fact_sales,121317,121317,true
dim_product,504,504,true
dim_sales_territory,10,10,true
dim_PurchaseOrderDetail,8845,8845,true
dim_PurchaseOrderHeader,4012,4012,true
dim_Vendor,104,104,true


Success: all 6 fact/dim tables loaded with the correct column names.
